In [38]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset,DataLoader

In [39]:
iris_dataset=load_iris()
X=iris_dataset.data
y=iris_dataset.target

In [40]:
X_train_val,X_test,y_train_val,y_test=train_test_split(X,y,test_size=0.15,stratify=y,random_state=42)

In [41]:
X_train,X_val,y_train,y_val=train_test_split(X_train_val,y_train_val,test_size=0.15,stratify=y_train_val,random_state=42)

In [42]:
scalar=StandardScaler()
X_train_scalinng=scalar.fit_transform(X_train)
X_val_scaling=scalar.transform(X_val)
X_test_scaling=scalar.transform(X_test)


In [43]:
X_train=torch.tensor(X_train_scalinng,dtype=torch.float32)
y_train=torch.tensor(y_train,dtype=torch.long)
X_val=torch.tensor(X_val_scaling,dtype=torch.float32)
y_val=torch.tensor(y_val,dtype=torch.long)
X_test=torch.tensor(X_test_scaling,dtype=torch.float32)
y_test=torch.tensor(y_test,dtype=torch.float32)


In [44]:
train_ds=TensorDataset(X_train,y_train)
val_ds=TensorDataset(X_val,y_val)
test_ds=TensorDataset(X_test,y_test)

In [45]:
train_loader=DataLoader(train_ds,batch_size=32,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=32,shuffle=False)
test_loader=DataLoader(test_ds,batch_size=32,shuffle=False)

In [46]:
class DNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(4,32),
            nn.ReLU(),
            nn.Linear(32,16),
            nn.ReLU(),
            nn.Linear(16,3)
        )

    def forward(self,x):
        return self.net(x)

In [47]:
model=DNN()

In [48]:
criterion=nn.CrossEntropyLoss()

In [49]:
optimizer=torch.optim.Adam(model.parameters(),lr=1E-3)

In [ ]:
def evaluate(loader):
    model.eval()
    total_loss=0
    correct=0
    total=0

    with torch.no_grad():
        for x_batch,y_batch in loader:
            output=model(x_batch)
            loss=criterion(output,y_batch)
            total_loss+=loss.item()
            preds=output.argmax(dim=1)
            correct+=(preds==y_batch).sum().item()
            total+=y_batch.size(0)
    accuracy=100.0*correct/total
    return total_loss/len(loader),accuracy

In [51]:
import copy
best_val_acc=0
best_weights=None
for epoch in range(50):
    model.train()
    running_loss=0
    for x_batch,y_batch in train_loader:
        optimizer.zero_grad() 
        output=model(x_batch)
        loss=criterion(output,y_batch)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()

    train_loss,train_acc=evaluate(train_loader)
    val_loss,val_acc=evaluate(val_loader)

    if val_acc>best_val_acc:
        best_val_acc=val_acc
        best_weights=copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1}] | [Train Acc: {train_acc:.2f}] | [Val Acc: {val_acc:.2f}]")

        

Epoch [1] | [Train Acc: 38.32] | [Val Acc: 35.00]
Epoch [2] | [Train Acc: 59.81] | [Val Acc: 60.00]
Epoch [3] | [Train Acc: 66.36] | [Val Acc: 65.00]
Epoch [4] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [5] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [6] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [7] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [8] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [9] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [10] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [11] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [12] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [13] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [14] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [15] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [16] | [Train Acc: 67.29] | [Val Acc: 65.00]
Epoch [17] | [Train Acc: 68.22] | [Val Acc: 65.00]
Epoch [18] | [Train Acc: 71.96] | [Val Acc: 65.00]
Epoch [19] | [Train Acc: 73.83] | [Val Acc: 65.00]
Epoch [20] | [Train Acc: 76.64] | [Val A